当前仓内已经有一个 `workflow observation e2e smoke`，并且已经跑通，但目前仍带有一段桥接代码：

def _make_executable_plan(plan: PlanSchema) -> PlanSchema:
    """
    The current ProtocolFactory stub returns non-executable circuits (e.g. strings).
    To validate the end-to-end workflow without changing business code, we replace
    each task's circuits with a minimal executable QuantumCircuit list.
    """

这说明当前链路虽然已经验证了：
- workflow 后半段可通
- observation 生成 / 存储 / 查询可通

但还没有做到真正原生的：

`ConfigSchema -> PlanBuilder.build_plan_from_config -> ProtocolFactory -> executable circuits -> run_plan -> analyze -> observation -> store -> query`

我现在希望你帮助我**逐步去掉这段“过桥代码”**，把当前 smoke 改成真正原生 e2e。

## 你的任务
请基于仓内真实实现，排查并回答以下问题：

### 1. 根因定位
请确认为什么当前 `PlanBuilder.build_plan_from_config(config)` 产出的 `plan.tasks[*].circuits` 不能直接被 `run_plan` 执行。
重点检查：
- `PlanBuilder`
- `ProtocolFactory`
- 各 protocol builder / generator
- `PlanSchema.tasks[*].circuits` 的真实类型
- `run_plan` / `Executor` 对 circuits 的类型要求

请明确指出：
- 当前 circuits 实际是什么（例如 string / placeholder / spec / IR / stub）
- `run_plan` 真正期望的是什么（例如 `QuantumCircuit`）
- 类型或语义是在什么环节断开的

### 2. 最小修复方案
请给出**改动最小、最符合当前架构意图**的修复方案，使得：
- `PlanBuilder.build_plan_from_config(config)` 产出的 plan
- 可以不经过 `_make_executable_plan()` 这类脚本内替换
- 直接交给 `run_plan(plan, executor)` 执行

要求：
- 不要大改业务架构
- 不要为了通过 smoke 去硬编码脚本特判
- 优先在最合理的真实生产点修复
- 保持 observation query smoke 与 workflow e2e smoke 的目标不变

### 3. 实施输出
请直接输出：
1. Root cause
2. Minimal fix strategy
3. 建议修改的文件路径
4. 完整补丁代码
5. 修改后的 `workflow_observation_e2e_smoke.py` 完整代码（去掉 `_make_executable_plan()`）
6. 运行命令
7. 验收标准

## 验收标准
最终目标是：
- `workflow_observation_e2e_smoke.py` 中不再需要 `_make_executable_plan()`
- `plan.tasks[*].circuits` 在 build plan 后就已经是 `QuantumCircuit`（或 `run_plan` 原生支持的可执行对象）
- `run_plan(plan, executor)` 可以直接执行
- analyze / observation / store / query 全链继续通过
- 脚本仍然保留当前分层打印输出
- 若某协议（例如 XEB）当前 builder 尚未完整实现，请优先选择仓内最容易真正产出 executable circuits 的协议，并说明理由

## 额外要求
- 如果根因是某个 protocol builder 仍是 stub，请优先做“最小真实化”修复，而不是继续在 smoke 脚本里兜底
- 如果仓内存在多个层次（spec -> compiled circuit -> executable circuit），请明确当前应在哪一层完成转换，避免职责错位
- 如果无法彻底修复，请至少给出一版“把桥接逻辑从脚本下沉到正确业务层”的最小方案，并说明这是过渡方案

## 输出风格
请不要长篇泛泛分析。
请严格按以下结构输出：
1. Root cause
2. Minimal fix strategy
3. Files to change
4. Patch
5. Updated smoke script
6. Run
7. Acceptance

In [5]:
import sys
from pathlib import Path


def add_repo_src_to_path() -> Path:
    """
    Find repo root by walking upward until src/egm exists,
    then prepend repo_root/src to sys.path.
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "egm").exists():
            src = p / "src"
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            return src
    raise RuntimeError("Could not find repo root containing src/egm")


src_path = add_repo_src_to_path()
print(f"Added src path: {src_path}")

Added src path: /Users/ousiachai/dev/errorgnomark_dev/src


In [6]:
from __future__ import annotations

from dataclasses import replace

from egm.analysis.xeb import analyze_task_execution_result
from egm.backends.base_backend import BaseBackend
from egm.circuits.circuit import QuantumCircuit
from egm.datastore.observation_store_memory import InMemoryObservationStore
from egm.execution.executor import Executor
from egm.execution.plan_runner import run_plan
from egm.schemas.configs import (
    ConfigBase,
    ConfigSchema,
    HardwareConfig,
    ProtocolBundle,
    ProtocolConfig,
)
from egm.schemas.plan import PlanSchema
from egm.services.planning.plan_builder import PlanBuilder
from egm.services.queries.observation_query_service import ObservationQueryService
from egm.services.serialization.observation_record import to_persistence_observation_dict
from egm.domain.records.task_observation_record import to_task_observation_record


class LocalCountsBackend(BaseBackend):
    """
    Minimal backend stub to avoid hardware dependencies.

    It returns deterministic-ish counts; the ideal reference probabilities are
    computed by Executor's internal IdealBackend.
    """

    def __init__(self) -> None:
        super().__init__(name="LocalCountsBackend")

    def run(self, circuit: QuantumCircuit, shots=None):
        n = circuit.num_qubits
        shots_i = int(shots or 10)
        return None, {("0" * n): shots_i}


def _print_header(
    *,
    protocol: str,
    backend_name: str,
    chip_name: str,
    qubits: list[int],
    depths: list[int],
    shots: int,
    repeats: int,
    plan_task_count: int | None,
) -> None:
    print("=" * 60)
    print("EGM Workflow E2E Smoke")
    print("=" * 60)
    print(f"protocol           : {protocol}")
    print(f"backend            : {backend_name}")
    print(f"chip               : {chip_name}")
    print(f"qubits             : {qubits}")
    print(f"depths             : {depths}")
    print(f"shots              : {shots}")
    print(f"repeats            : {repeats}")
    if plan_task_count is not None:
        print(f"plan task count    : {plan_task_count}")
    print("=" * 60)
    print()


def _stage(i: int, total: int, msg: str) -> None:
    print(f"[{i}/{total}] {msg} ...")


def _ok(msg: str) -> None:
    print(f"[ok ] {msg}")


def _warn(msg: str) -> None:
    print(f"[warn] {msg}")


def _make_executable_plan(plan: PlanSchema) -> PlanSchema:
    """
    The current ProtocolFactory stub returns non-executable circuits (e.g. strings).
    To validate the end-to-end workflow without changing business code, we replace
    each task's circuits with a minimal executable QuantumCircuit list.
    """

    new_tasks = []
    for task in plan.tasks:
        qc = QuantumCircuit(qubits=task.qubits)
        new_tasks.append(replace(task, circuits=[qc]))
    return replace(plan, tasks=new_tasks)


def main() -> None:
    total_stages = 7

    protocol = "XEB"
    backend_name = "LocalCountsBackend"
    chip_name = "chip-alpha"
    qubits = [0, 1]
    depths = [3]
    shots = 10
    repeats = 1

    _print_header(
        protocol=protocol,
        backend_name=backend_name,
        chip_name=chip_name,
        qubits=qubits,
        depths=depths,
        shots=shots,
        repeats=repeats,
        plan_task_count=None,
    )

    _stage(1, total_stages, "building config")
    config = ConfigSchema(
        base=ConfigBase(plan_id="plan-e2e-1", backend_name=backend_name),
        hardware=HardwareConfig(chip_name=chip_name, gate_set="native", noise_flags={}),
        protocol=ProtocolConfig(
            number_of_circuits=repeats,
            shots=shots,
            bundles=[ProtocolBundle(protocol=protocol, qubits=[qubits], depths=depths)],
        ),
    )
    _ok("config built")

    _stage(2, total_stages, "building plan")
    plan = PlanBuilder.build_plan_from_config(config)
    assert plan.tasks, "Expected non-empty plan.tasks"
    _ok(f"plan built: {len(plan.tasks)} task(s)")

    # Make plan executable for run_plan's QuantumCircuit type gate.
    plan = _make_executable_plan(plan)
    assert all(isinstance(c, QuantumCircuit) for t in plan.tasks for c in t.circuits)

    # Task summary (first 3 tasks)
    for idx, task in enumerate(plan.tasks[:3], start=1):
        t_depth = task.meta_data.get("depth")
        t_shots = task.meta_data.get("shots")
        t_backend = task.meta_data.get("backend_name")
        t_chip = task.meta_data.get("chip_name")
        print()
        print(f"[task {idx}/{len(plan.tasks)}]")
        print(f"  task_id          : {task.task_id}")
        print(f"  protocol         : {task.protocol}")
        if t_backend is not None:
            print(f"  backend          : {t_backend}")
        if t_chip is not None:
            print(f"  chip             : {t_chip}")
        print(f"  qubits           : {task.qubits}")
        if t_depth is not None:
            print(f"  depth            : {t_depth}")
        if t_shots is not None:
            print(f"  shots            : {t_shots}")
        print(f"  repeats          : {repeats}")

    print()
    _stage(3, total_stages, "running plan")
    executor = Executor(LocalCountsBackend())
    exec_res = run_plan(plan, executor)
    assert exec_res.task_results, "Expected non-empty task_results"
    _ok(f"tasks executed: {len(exec_res.task_results)}")

    # execution status summary
    status_counts: dict[str, int] = {}
    for tr in exec_res.task_results:
        status_counts[tr.status] = status_counts.get(tr.status, 0) + 1

    print()
    _stage(4, total_stages, "analyzing results")
    observations = []
    warned_spb_fit = False
    for task, tr in zip(plan.tasks, exec_res.task_results):
        ar = analyze_task_execution_result(task, tr)
        obs = to_task_observation_record(task, tr, ar)
        observations.append(obs)

        # Friendly warning summary (best-effort)
        payload = ar.analysis_payload or {}
        spb_fit = (
            payload.get("spb_analysis", {}) if isinstance(payload, dict) else {}
        )
        spb_fit_results = (
            spb_fit.get("fit_results", {}) if isinstance(spb_fit, dict) else {}
        )
        msg = spb_fit_results.get("message")
        if isinstance(msg, str) and "Insufficient points (<3)" in msg:
            warned_spb_fit = True

    assert observations, "Expected at least one TaskObservationRecord"
    if warned_spb_fit:
        _warn("SPB fit skipped because insufficient points (<3)")
    _ok(f"observations built: {len(observations)}")

    print()
    _stage(5, total_stages, "converting observations")
    persistence_dicts = [to_persistence_observation_dict(o) for o in observations]
    assert persistence_dicts, "Expected at least one persistence dict"
    _ok(f"persistence payloads built: {len(persistence_dicts)}")

    print()
    _stage(6, total_stages, "saving to store")
    store = InMemoryObservationStore()
    svc = ObservationQueryService(store=store)
    ids = [store.save_observation(d) for d in persistence_dicts]
    assert ids, "Expected at least one observation_id"
    _ok(f"observations saved: {len(ids)}")

    print()
    _stage(7, total_stages, "querying observations")
    got0 = svc.get_observation(ids[0])
    assert got0 is not None, "Expected get_observation hit"
    _ok("get_observation hit")
    assert got0.get("task_id"), "Expected task_id in observation"
    assert got0.get("protocol"), "Expected protocol in observation"
    assert got0.get("backend_name"), "Expected backend_name in observation"

    all_obs = svc.list_observations()
    assert len(all_obs) > 0, "Expected non-empty list_observations"
    _ok(f"list_observations returned {len(all_obs)} item(s)")

    filtered = svc.filter_observations(protocol=got0["protocol"])
    assert filtered, "Expected non-empty filter_observations(protocol=...)"
    _ok(
        f"filter_observations(protocol='{got0['protocol']}') returned {len(filtered)} item(s)"
    )

    missing = svc.get_observation("does-not-exist")
    assert missing is None, "Expected missing id -> None"

    print()
    print("[result summary]")
    print(f"  tasks executed   : {len(exec_res.task_results)}")
    print(f"  observations     : {len(observations)}")
    print(f"  saved            : {len(ids)}")
    print(f"  execution status : {status_counts}")
    print()
    print("[sample observation]")
    sample_id = ids[0]
    print(f"  id               : {sample_id}")
    print(f"  task_id          : {got0.get('task_id')}")
    print(f"  protocol         : {got0.get('protocol')}")
    print(f"  backend          : {got0.get('backend_name')}")
    print(f"  chip             : {got0.get('chip_name')}")
    print(f"  qubits           : {got0.get('qubits')}")
    print()
    print("workflow observation e2e smoke passed")


if __name__ == "__main__":
    main()



[INFO] QuantumEngine initialized with backend 'LocalCountsBackend'.
[INFO] Executing 1 circuits (10 shots each) on backend 'LocalCountsBackend'.
[INFO] All 1 circuits executed.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.


EGM Workflow E2E Smoke
protocol           : XEB
backend            : LocalCountsBackend
chip               : chip-alpha
qubits             : [0, 1]
depths             : [3]
shots              : 10
repeats            : 1

[1/7] building config ...
[ok ] config built
[2/7] building plan ...
[ok ] plan built: 1 task(s)

[task 1/1]
  task_id          : 44c0283f-ff9f-4986-aaa8-a75330e2a30a
  protocol         : XEB
  backend          : LocalCountsBackend
  chip             : chip-alpha
  qubits           : [0, 1]
  depth            : 3
  shots            : 10
  repeats          : 1

[3/7] running plan ...
[ok ] tasks executed: 1

[4/7] analyzing results ...
[warn] SPB fit skipped because insufficient points (<3)
[ok ] observations built: 1

[5/7] converting observations ...
[ok ] persistence payloads built: 1

[6/7] saving to store ...
[ok ] observations saved: 1

[7/7] querying observations ...
[ok ] get_observation hit
[ok ] list_observations returned 1 item(s)
[ok ] filter_observations(pr

In [ ]:
# ### Optional: matrix noise simulator (`DummyBackend` / `UnifiedMatrixBackend`)

# Run the **“Added src path”** cell first so `import egm` works, then run the **next** cell to execute the same workflow with `egm.backends.dummy_backend_xeb.DummyBackend` (sampled noisy counts from a matrix + depolarization model).

# The default smoke cell below still uses `LocalCountsBackend` for speed and determinism.

Added src path: /Users/ousiachai/dev/errorgnomark_dev/src


### 矩阵噪声版 E2E（本格下方 code）

下面一格在 **同一套链路** 上把 `Executor` 里的「有噪计数」从 `LocalCountsBackend`（全落在 `0…0` 的占位）换成 **`DummyBackend` / `UnifiedMatrixBackend`**：用门矩阵做态演化，再叠加退极化 / 采样得到 **更接近模拟器的计数字典**；理想参考仍由 `Executor` 内部的 `IdealBackend` 单独计算。

**和下一格的关系**：再下一格是 **默认 smoke**（`LocalCountsBackend`）。本 code 格的 **终端打印版式** 与那一格对齐（同一套 `_print_header` / `_stage` / `_ok` / 任务摘要 / `[result summary]` / `[sample observation]`）；**仅** `backend` 行会变成 `UnifiedMatrixBackend`，有噪计数来自矩阵噪声模型。请先运行上面的 **「Added src path」** 那一格。


In [9]:
from __future__ import annotations

import sys
from pathlib import Path

# Ensure `egm` import works when this cell is run alone (idempotent).
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "egm").exists():
        _src = str(_root / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        break

from egm.analysis.xeb import analyze_task_execution_result
from egm.backends.dummy_backend_xeb import DummyBackend
from egm.circuits.circuit import QuantumCircuit
from egm.datastore.observation_store_memory import InMemoryObservationStore
from egm.execution.executor import Executor
from egm.execution.plan_runner import run_plan
from egm.schemas.configs import (
    ConfigBase,
    ConfigSchema,
    HardwareConfig,
    ProtocolBundle,
    ProtocolConfig,
)
from egm.services.planning.plan_builder import PlanBuilder
from egm.services.queries.observation_query_service import ObservationQueryService
from egm.services.serialization.observation_record import to_persistence_observation_dict
from egm.domain.records.task_observation_record import to_task_observation_record


def main_matrix_noise_smoke() -> None:
    """Same chain as default smoke: plan → run_plan → analyze → observation → store → query,
    but Executor uses matrix-based noisy counts from DummyBackend (seeded RNG).
    """
    total_stages = 7
    noise = DummyBackend(seed=42)
    backend_name = noise.name
    protocol = "XEB"
    chip_name = "chip-alpha"
    qubits = [0, 1]
    depths = [30]
    shots = 1000
    repeats = 10

    print("=" * 60)
    print("EGM Workflow E2E Smoke (matrix noise: DummyBackend)")
    print("=" * 60)
    print(f"protocol           : {protocol}")
    print(f"backend            : {backend_name}")
    print(f"chip               : {chip_name}")
    print(f"qubits             : {qubits}")
    print(f"depths             : {depths}")
    print(f"shots              : {shots}")
    print(f"repeats            : {repeats}")
    print("=" * 60)
    print()

    print(f"[1/{total_stages}] building config ...")
    config = ConfigSchema(
        base=ConfigBase(plan_id="plan-e2e-matrix-noise-1", backend_name=backend_name),
        hardware=HardwareConfig(chip_name=chip_name, gate_set="native", noise_flags={}),
        protocol=ProtocolConfig(
            number_of_circuits=repeats,
            shots=shots,
            bundles=[ProtocolBundle(protocol=protocol, qubits=[qubits], depths=depths)],
        ),
    )
    print("[ok ] config built")

    print(f"[2/{total_stages}] building plan ...")
    plan = PlanBuilder.build_plan_from_config(config)
    assert plan.tasks
    assert all(isinstance(c, QuantumCircuit) for t in plan.tasks for c in t.circuits)
    print(f"[ok ] plan built: {len(plan.tasks)} task(s)")

    print(f"[3/{total_stages}] running plan ...")
    exec_res = run_plan(plan, Executor(noise))
    assert exec_res.task_results
    print(f"[ok ] tasks executed: {len(exec_res.task_results)}")

    print(f"[4/{total_stages}] analyzing + building observations ...")
    observations = []
    for task, tr in zip(plan.tasks, exec_res.task_results):
        ar = analyze_task_execution_result(task, tr)
        observations.append(to_task_observation_record(task, tr, ar))
    print(f"[ok ] observations built: {len(observations)}")

    print(f"[5/{total_stages}] converting to persistence dicts ...")
    persistence_dicts = [to_persistence_observation_dict(o) for o in observations]
    print(f"[ok ] payloads: {len(persistence_dicts)}")

    print(f"[6/{total_stages}] saving ...")
    store = InMemoryObservationStore()
    svc = ObservationQueryService(store=store)
    ids = [store.save_observation(d) for d in persistence_dicts]
    print(f"[ok ] saved: {len(ids)}")

    print(f"[7/{total_stages}] querying ...")
    got0 = svc.get_observation(ids[0])
    assert got0 is not None
    assert svc.filter_observations(protocol=got0["protocol"])
    print("[ok ] query round-trip")

    print()
    print("matrix-noise workflow observation e2e smoke passed")
    print(f"  backend_name in record: {got0.get('backend_name')}")
    print(f"  execution_summary     : {got0.get('execution_summary')}")


main_matrix_noise_smoke()


[INFO] QuantumEngine initialized with backend 'UnifiedMatrixBackend'.
[INFO] Executing 10 circuits (1000 shots each) on backend 'UnifiedMatrixBackend'.
[INFO] All 10 circuits executed.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.


EGM Workflow E2E Smoke (matrix noise: DummyBackend)
protocol           : XEB
backend            : UnifiedMatrixBackend
chip               : chip-alpha
qubits             : [0, 1]
depths             : [30]
shots              : 1000
repeats            : 10

[1/7] building config ...
[ok ] config built
[2/7] building plan ...
[ok ] plan built: 1 task(s)
[3/7] running plan ...
[ok ] tasks executed: 1
[4/7] analyzing + building observations ...
[ok ] observations built: 1
[5/7] converting to persistence dicts ...
[ok ] payloads: 1
[6/7] saving ...
[ok ] saved: 1
[7/7] querying ...
[ok ] query round-trip

matrix-noise workflow observation e2e smoke passed
  backend_name in record: UnifiedMatrixBackend
  execution_summary     : {'num_pairs': 10, 'pairs_digest': None, 'artifact_ref': None}


[INFO] QuantumEngine initialized with backend 'LocalCountsBackend'.
[INFO] Executing 1 circuits (10 shots each) on backend 'LocalCountsBackend'.
noisy counts 是fake.

In [8]:
from __future__ import annotations

from egm.analysis.xeb import analyze_task_execution_result
from egm.backends.base_backend import BaseBackend
from egm.circuits.circuit import QuantumCircuit
from egm.datastore.observation_store_memory import InMemoryObservationStore
from egm.execution.executor import Executor
from egm.execution.plan_runner import run_plan
from egm.schemas.configs import (
    ConfigBase,
    ConfigSchema,
    HardwareConfig,
    ProtocolBundle,
    ProtocolConfig,
)
from egm.services.planning.plan_builder import PlanBuilder
from egm.services.queries.observation_query_service import ObservationQueryService
from egm.services.serialization.observation_record import to_persistence_observation_dict
from egm.domain.records.task_observation_record import to_task_observation_record


class LocalCountsBackend(BaseBackend):
    """
    Minimal backend stub to avoid hardware dependencies.

    It returns deterministic-ish counts; the ideal reference probabilities are
    computed by Executor's internal IdealBackend.
    """

    def __init__(self) -> None:
        super().__init__(name="LocalCountsBackend")

    def run(self, circuit: QuantumCircuit, shots=None):
        n = circuit.num_qubits
        shots_i = int(shots or 10)
        return None, {("0" * n): shots_i}


def _print_header(
    *,
    protocol: str,
    backend_name: str,
    chip_name: str,
    qubits: list[int],
    depths: list[int],
    shots: int,
    repeats: int,
    plan_task_count: int | None,
) -> None:
    print("=" * 60)
    print("EGM Workflow E2E Smoke")
    print("=" * 60)
    print(f"protocol           : {protocol}")
    print(f"backend            : {backend_name}")
    print(f"chip               : {chip_name}")
    print(f"qubits             : {qubits}")
    print(f"depths             : {depths}")
    print(f"shots              : {shots}")
    print(f"repeats            : {repeats}")
    if plan_task_count is not None:
        print(f"plan task count    : {plan_task_count}")
    print("=" * 60)
    print()


def _stage(i: int, total: int, msg: str) -> None:
    print(f"[{i}/{total}] {msg} ...")


def _ok(msg: str) -> None:
    print(f"[ok ] {msg}")


def _warn(msg: str) -> None:
    print(f"[warn] {msg}")


def main() -> None:
    total_stages = 7

    protocol = "XEB"
    backend_name = "LocalCountsBackend"
    chip_name = "chip-alpha"
    qubits = [0, 1]
    depths = [3]
    shots = 10
    repeats = 1

    _print_header(
        protocol=protocol,
        backend_name=backend_name,
        chip_name=chip_name,
        qubits=qubits,
        depths=depths,
        shots=shots,
        repeats=repeats,
        plan_task_count=None,
    )

    _stage(1, total_stages, "building config")
    config = ConfigSchema(
        base=ConfigBase(plan_id="plan-e2e-1", backend_name=backend_name),
        hardware=HardwareConfig(chip_name=chip_name, gate_set="native", noise_flags={}),
        protocol=ProtocolConfig(
            number_of_circuits=repeats,
            shots=shots,
            bundles=[ProtocolBundle(protocol=protocol, qubits=[qubits], depths=depths)],
        ),
    )
    _ok("config built")

    _stage(2, total_stages, "building plan")
    plan = PlanBuilder.build_plan_from_config(config)
    assert plan.tasks, "Expected non-empty plan.tasks"
    _ok(f"plan built: {len(plan.tasks)} task(s)")
    assert all(isinstance(c, QuantumCircuit) for t in plan.tasks for c in t.circuits)

    # Task summary (first 3 tasks)
    for idx, task in enumerate(plan.tasks[:3], start=1):
        t_depth = task.meta_data.get("depth")
        t_shots = task.meta_data.get("shots")
        t_backend = task.meta_data.get("backend_name")
        t_chip = task.meta_data.get("chip_name")
        print()
        print(f"[task {idx}/{len(plan.tasks)}]")
        print(f"  task_id          : {task.task_id}")
        print(f"  protocol         : {task.protocol}")
        if t_backend is not None:
            print(f"  backend          : {t_backend}")
        if t_chip is not None:
            print(f"  chip             : {t_chip}")
        print(f"  qubits           : {task.qubits}")
        if t_depth is not None:
            print(f"  depth            : {t_depth}")
        if t_shots is not None:
            print(f"  shots            : {t_shots}")
        print(f"  repeats          : {repeats}")

    print()
    _stage(3, total_stages, "running plan")
    executor = Executor(LocalCountsBackend())
    exec_res = run_plan(plan, executor)
    assert exec_res.task_results, "Expected non-empty task_results"
    _ok(f"tasks executed: {len(exec_res.task_results)}")

    # execution status summary
    status_counts: dict[str, int] = {}
    for tr in exec_res.task_results:
        status_counts[tr.status] = status_counts.get(tr.status, 0) + 1

    print()
    _stage(4, total_stages, "analyzing results")
    observations = []
    warned_spb_fit = False
    for task, tr in zip(plan.tasks, exec_res.task_results):
        ar = analyze_task_execution_result(task, tr)
        obs = to_task_observation_record(task, tr, ar)
        observations.append(obs)

        # Friendly warning summary (best-effort)
        payload = ar.analysis_payload or {}
        spb_fit = payload.get("spb_analysis", {}) if isinstance(payload, dict) else {}
        spb_fit_results = spb_fit.get("fit_results", {}) if isinstance(spb_fit, dict) else {}
        msg = spb_fit_results.get("message")
        if isinstance(msg, str) and "Insufficient points (<3)" in msg:
            warned_spb_fit = True

    assert observations, "Expected at least one TaskObservationRecord"
    if warned_spb_fit:
        _warn("SPB fit skipped because insufficient points (<3)")
    _ok(f"observations built: {len(observations)}")

    print()
    _stage(5, total_stages, "converting observations")
    persistence_dicts = [to_persistence_observation_dict(o) for o in observations]
    assert persistence_dicts, "Expected at least one persistence dict"
    _ok(f"persistence payloads built: {len(persistence_dicts)}")

    print()
    _stage(6, total_stages, "saving to store")
    store = InMemoryObservationStore()
    svc = ObservationQueryService(store=store)
    ids = [store.save_observation(d) for d in persistence_dicts]
    assert ids, "Expected at least one observation_id"
    _ok(f"observations saved: {len(ids)}")

    print()
    _stage(7, total_stages, "querying observations")
    got0 = svc.get_observation(ids[0])
    assert got0 is not None, "Expected get_observation hit"
    _ok("get_observation hit")
    assert got0.get("task_id"), "Expected task_id in observation"
    assert got0.get("protocol"), "Expected protocol in observation"
    assert got0.get("backend_name"), "Expected backend_name in observation"

    all_obs = svc.list_observations()
    assert len(all_obs) > 0, "Expected non-empty list_observations"
    _ok(f"list_observations returned {len(all_obs)} item(s)")

    filtered = svc.filter_observations(protocol=got0["protocol"])
    assert filtered, "Expected non-empty filter_observations(protocol=...)"
    _ok(
        f"filter_observations(protocol='{got0['protocol']}') returned {len(filtered)} item(s)"
    )

    missing = svc.get_observation("does-not-exist")
    assert missing is None, "Expected missing id -> None"

    print()
    print("[result summary]")
    print(f"  tasks executed   : {len(exec_res.task_results)}")
    print(f"  observations     : {len(observations)}")
    print(f"  saved            : {len(ids)}")
    print(f"  execution status : {status_counts}")
    print()
    print("[sample observation]")
    sample_id = ids[0]
    print(f"  id               : {sample_id}")
    print(f"  task_id          : {got0.get('task_id')}")
    print(f"  protocol         : {got0.get('protocol')}")
    print(f"  backend          : {got0.get('backend_name')}")
    print(f"  chip             : {got0.get('chip_name')}")
    print(f"  qubits           : {got0.get('qubits')}")
    print()
    print("workflow observation e2e smoke passed")


if __name__ == "__main__":
    main()

[INFO] QuantumEngine initialized with backend 'LocalCountsBackend'.
[INFO] Executing 1 circuits (10 shots each) on backend 'LocalCountsBackend'.
[INFO] All 1 circuits executed.
[WARNING] [SPB-FIT] Insufficient points (<3) for SPB fit.


EGM Workflow E2E Smoke
protocol           : XEB
backend            : LocalCountsBackend
chip               : chip-alpha
qubits             : [0, 1]
depths             : [3]
shots              : 10
repeats            : 1

[1/7] building config ...
[ok ] config built
[2/7] building plan ...
[ok ] plan built: 1 task(s)

[task 1/1]
  task_id          : cbe1b017-d701-4509-86ca-9e936871f565
  protocol         : XEB
  backend          : LocalCountsBackend
  chip             : chip-alpha
  qubits           : [0, 1]
  depth            : 3
  shots            : 10
  repeats          : 1

[3/7] running plan ...
[ok ] tasks executed: 1

[4/7] analyzing results ...
[warn] SPB fit skipped because insufficient points (<3)
[ok ] observations built: 1

[5/7] converting observations ...
[ok ] persistence payloads built: 1

[6/7] saving to store ...
[ok ] observations saved: 1

[7/7] querying observations ...
[ok ] get_observation hit
[ok ] list_observations returned 1 item(s)
[ok ] filter_observations(pr

In [ ]:
from egm.services.planning.plan_builder import PlanBuilder
from egm.execution.executor import Executor
from egm.execution.plan_runner import run_plan
from egm.analysis.xeb import analyze_task_execution_result
from egm.domain.records.task_observation_record import to_task_observation_record
from egm.services.serialization.observation_record import to_persistence_observation_dict
from egm.datastore.observation_store_memory import InMemoryObservationStore
from egm.services.queries.observation_query_service import ObservationQueryService

config = build_config(...)                               # 1. 输入实验配置
plan = PlanBuilder.build_plan_from_config(config)        # 2. 配置 -> 实验计划

executor = Executor(LocalBackend())
exec_result = run_plan(plan, executor)                   # 3. 执行实验计划

observations = []
for task, result in zip(plan.tasks, exec_result.task_results):
    analysis = analyze_task_execution_result(task, result)    # 4. 分析执行结果
    observation = to_task_observation_record(task, result, analysis)
    observations.append(observation)                          # 5. 生成观测记录

payloads = [to_persistence_observation_dict(o) for o in observations]  # 6. 转存储格式

store = InMemoryObservationStore()
query_service = ObservationQueryService(store)

ids = [store.save_observation(p) for p in payloads]     # 7. 保存观测数据
result = query_service.get_observation(ids[0])          # 8. 查询结果